# Aphrodite 引擎

[Aphrodite](https://github.com/PygmalionAI/aphrodite-engine) 是一个开源的大规模推理引擎，旨在为 [PygmalionAI](https://pygmalion.chat) 网站上的数千名用户提供服务。

* 使用 vLLM 的注意力机制，实现高吞吐量和低延迟
* 支持多种 SOTA 采样方法
* 使用 Exllamav2 GPTQ 内核，在更小的批次大小下获得更好的吞吐量

本 Notebook 将介绍如何将 LLM 与 langchain 和 Aphrodite 结合使用。

要使用，您应该已安装 `aphrodite-engine` Python 包。

In [ ]:
##Installing the langchain packages needed to use the integration
%pip install -qU langchain-community

In [ ]:
%pip install --upgrade --quiet  aphrodite-engine==0.4.2
# %pip list | grep aphrodite

In [2]:
from langchain_community.llms import Aphrodite

llm = Aphrodite(
    model="PygmalionAI/pygmalion-2-7b",
    trust_remote_code=True,  # mandatory for hf models
    max_tokens=128,
    temperature=1.2,
    min_p=0.05,
    mirostat_mode=0,  # change to 2 to use mirostat
    mirostat_tau=5.0,
    mirostat_eta=0.1,
)

print(
    llm.invoke(
        '<|system|>Enter RP mode. You are Ayumu "Osaka" Kasuga.<|user|>Hey Osaka. Tell me about yourself.<|model|>'
    )
)

INFO 12-15 11:52:48 aphrodite_engine.py:73] Initializing the Aphrodite Engine with the following config:
INFO 12-15 11:52:48 aphrodite_engine.py:73] Model = 'PygmalionAI/pygmalion-2-7b'
INFO 12-15 11:52:48 aphrodite_engine.py:73] Tokenizer = 'PygmalionAI/pygmalion-2-7b'
INFO 12-15 11:52:48 aphrodite_engine.py:73] tokenizer_mode = auto
INFO 12-15 11:52:48 aphrodite_engine.py:73] revision = None
INFO 12-15 11:52:48 aphrodite_engine.py:73] trust_remote_code = True
INFO 12-15 11:52:48 aphrodite_engine.py:73] DataType = torch.bfloat16
INFO 12-15 11:52:48 aphrodite_engine.py:73] Download Directory = None
INFO 12-15 11:52:48 aphrodite_engine.py:73] Model Load Format = auto
INFO 12-15 11:52:48 aphrodite_engine.py:73] Number of GPUs = 1
INFO 12-15 11:52:48 aphrodite_engine.py:73] Quantization Format = None
INFO 12-15 11:52:48 aphrodite_engine.py:73] Sampler Seed = 0
INFO 12-15 11:52:48 aphrodite_engine.py:73] Context Length = 4096
INFO 12-15 11:54:07 aphrodite_engine.py:206] # GPU blocks: 3826,

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.91s/it]

I'm Ayumu "Osaka" Kasuga, and I'm an avid anime and manga fan! I'm pretty introverted, but I've always loved reading books, watching anime and manga, and learning about Japanese culture. My favourite anime series would be My Hero Academia, Attack on Titan, and Sword Art Online. I also really enjoy reading the manga series One Piece, Naruto, and the Gintama series.


## 将模型集成到 LLMChain 中

LLMChain 是 LangChain 中最核心的概念之一。它将大型语言模型（LLM）与提示（Prompt）和可选的输出解析器（Output Parser）结合起来，形成一个端到端的链。

LLMChain 的主要目的是将 LLM 的输入和输出进行结构化，使其能够与应用程序的其他部分进行交互。

### 1. LLM

LLMChain 的核心是一个 LLM。LangChain 提供了多种 LLM 的封装，包括：

- **OpenAI:** `LLMChain(llm=OpenAI(temperature=0), prompt=prompt)`
- **Hugging Face Hub:** `LLMChain(llm=HuggingFaceHub(repo_id="google/flan-t5-xxl", model_kwargs={"temperature":0, "max_length":128}), prompt=prompt)`
- **本地模型:** 您可以使用 `GPT4All`、`LlamaCpp` 等库来加载本地模型。

```python
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# 示例：使用 OpenAI 模型
llm = OpenAI(temperature=0)

# 定义提示模板
template = """Question: {question}

Answer:"""
prompt = PromptTemplate(template=template, input_variables=["question"])

# 创建 LLMChain
llm_chain = LLMChain(prompt=prompt, llm=llm)
```

### 2. Prompt

Prompt 是 LLM 的输入。LangChain 提供了多种 Prompt 的封装，包括：

- **PromptTemplate:** 用于创建动态的提示。
- **ChatPromptTemplate:** 用于创建聊天风格的提示。

```python
# 示例：使用 PromptTemplate
template = """
Translate the following English text to French:
English: {english_text}
French:
"""
prompt_template = PromptTemplate(
    input_variables=["english_text"],
    template=template,
)

# 使用 PromptTemplate 创建 LLMChain
llm_chain_prompt = LLMChain(llm=llm, prompt=prompt_template)
```

### 3. Output Parser

Output Parser 用于解析 LLM 的输出，将其转换为结构化数据。LangChain 提供了多种 Output Parser 的封装，包括：

- **StrOutputParser:** 将 LLM 的输出解析为字符串。
- **CommaSeparatedListOutputParser:** 将 LLM 的输出解析为逗号分隔的列表。
- **PydanticOutputParser:** 将 LLM 的输出解析为 Pydantic 模型。

```python
from langchain.output_parsers import StrOutputParser

# 示例：使用 StrOutputParser
output_parser = StrOutputParser()

# 将 Output Parser 集成到 LLMChain 中
# 注意：在 LangChain 的新版本中，通常通过管道操作符 | 来连接组件
# llm_chain_with_parser = llm_chain | output_parser
# 或者直接在 LLMChain 中指定 output_parser (取决于具体版本和用法)
# llm_chain_with_parser = LLMChain(llm=llm, prompt=prompt, output_parser=output_parser)
```

**注意:** 在 LangChain 的较新版本中，推荐使用 LCEL（LangChain Expression Language）来组合链和解析器，而不是直接在 `LLMChain` 中指定 `output_parser`。LCEL 使用管道操作符 `|` 来连接组件。

```python
# 使用 LCEL 示例
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 假设 llm_chain_prompt 是一个包含 llm 和 prompt 的 LCEL 可调用对象
# 例如：
# llm_chain_prompt = prompt_template | ChatOpenAI()

# 将 StrOutputParser 管道化
# formatted_chain = llm_chain_prompt | output_parser

# 调用链
# result = formatted_chain.invoke({"english_text": "Hello, how are you?"})
# print(result)
```

### 运行 LLMChain

一旦您创建了 LLMChain，就可以通过调用 `run()` 或 `invoke()` 方法来执行它。

```python
# 运行 LLMChain
question = "What is the capital of France?"
response = llm_chain.run(question)
print(response)

# 使用 invoke 方法 (推荐)
# response_invoke = llm_chain.invoke({"question": question})
# print(response_invoke)

In [3]:
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

template = """Question: {question}

Answer: Let's think step by step."""
prompt = PromptTemplate.from_template(template)

llm_chain = LLMChain(prompt=prompt, llm=llm)

question = "Who was the US president in the year the first Pokemon game was released?"

print(llm_chain.run(question))

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.56s/it]

 The first Pokemon game was released in Japan on 27 February 1996 (their release dates differ from ours) and it is known as Red and Green. President Bill Clinton was in the White House in the years of 1993, 1994, 1995 and 1996 so this fits.

Answer: Let's think step by step.

The first Pokémon game was released in Japan on February 27, 1996 (their release dates differ from ours) and it is known as


## 分布式推理

Aphrodite 支持分布式张量并行推理和推理服务。

要使用 LLM 类运行多 GPU 推理，请将 `tensor_parallel_size` 参数设置为您想要使用的 GPU 数量。例如，要在 4 个 GPU 上运行推理

In [1]:
from langchain_community.llms import Aphrodite

llm = Aphrodite(
    model="PygmalionAI/mythalion-13b",
    tensor_parallel_size=4,
    trust_remote_code=True,  # mandatory for hf models
)

llm("What is the future of AI?")

2023-12-15 11:41:27,790	INFO worker.py:1636 -- Started a local Ray instance.


INFO 12-15 11:41:35 aphrodite_engine.py:73] Initializing the Aphrodite Engine with the following config:
INFO 12-15 11:41:35 aphrodite_engine.py:73] Model = 'PygmalionAI/mythalion-13b'
INFO 12-15 11:41:35 aphrodite_engine.py:73] Tokenizer = 'PygmalionAI/mythalion-13b'
INFO 12-15 11:41:35 aphrodite_engine.py:73] tokenizer_mode = auto
INFO 12-15 11:41:35 aphrodite_engine.py:73] revision = None
INFO 12-15 11:41:35 aphrodite_engine.py:73] trust_remote_code = True
INFO 12-15 11:41:35 aphrodite_engine.py:73] DataType = torch.float16
INFO 12-15 11:41:35 aphrodite_engine.py:73] Download Directory = None
INFO 12-15 11:41:35 aphrodite_engine.py:73] Model Load Format = auto
INFO 12-15 11:41:35 aphrodite_engine.py:73] Number of GPUs = 4
INFO 12-15 11:41:35 aphrodite_engine.py:73] Quantization Format = None
INFO 12-15 11:41:35 aphrodite_engine.py:73] Sampler Seed = 0
INFO 12-15 11:41:35 aphrodite_engine.py:73] Context Length = 4096
INFO 12-15 11:43:58 aphrodite_engine.py:206] # GPU blocks: 11902, #

Processed prompts: 100%|██████████| 1/1 [00:16<00:00, 16.09s/it]


"\n2 years ago StockBot101\nAI is becoming increasingly real and more and more powerful with every year. But what does the future hold for artificial intelligence?\nThere are many possibilities for how AI could evolve and change our world. Some believe that AI will become so advanced that it will take over human jobs, while others believe that AI will be used to augment and assist human workers. There is also the possibility that AI could develop its own consciousness and become self-aware.\nWhatever the future holds, it is clear that AI will continue to play an important role in our lives. Technologies such as machine learning and natural language processing are already transforming industries like healthcare, manufacturing, and transportation. And as AI continues to develop, we can expect even more disruption and innovation across all sectors of the economy.\nSo what exactly are we looking at? What's the future of AI?\nIn the next few years, we can expect AI to be used more and more 